# Chapter 10: Epidemiological Models

> This chapter compares the infection and removal rules of mechanistic SIR with autoregressive forecasts that update predicted counts from recent observations, and shows how $R_t=\beta S(t)/\gamma$ determines whether infections grow while immunity above $1-1/R_0$ makes a small introduction decline.

:::{note} Running the code
The Python cells on this page run **in your browser**. Click the **power icon** at the top of the page to activate the kernel, then run — or edit — any cell. Packages (NumPy, SciPy, …) load automatically the first time you import them (a few seconds).
:::

## Motivation

An epidemic is a spreading phenomenon: individuals have states, and interactions can change those states. Suppose one infectious person enters a population in which almost everyone can catch the disease. Will infections grow? If so, when will they stop growing? A mechanistic model starts with rules for those state changes and derives an epidemic curve. We will look at the most famous of such models: the **SIR model**, introduced by Kermack and McKendrick in 1927.

A **statistical model** can use observations directly to predict the near-term future, without specifying how an infection passes between people. We will study linear and exponential autoregressive forecast rules that update each predicted count from its preceding value.

SIR has three state variables and two parameters, yet it gives a threshold for initial growth, shows that the growth is exponential, and shows that infections later naturally decline as each infection reduces the number still susceptible. The basic reproduction number $R_0=\beta N/\gamma$ summarizes the early behavior and provides a threshold for exponential growth.

The statistical forecasts have no analogous susceptible compartment or reproduction number. Their parameters describe a pattern in observations, but they can be extremely effective and efficient for short-term accurate predictions.

## Two classes of models

- **Mechanistic models** specify the states and the events that change them. Their equations express assumptions about transmission and removal. A compartmental SIR model is one example.
- **Statistical models** estimate a relationship from observed counts and use it to forecast later counts. An autoregressive rule uses earlier values of a series to predict its next value. It need not contain, or be derived from, SIR compartments.

A model may combine both ideas, for example by fitting a mechanistic transmission process to noisy reported counts. Here we study each class on its own so that the meaning of its parameters and predictions is clear.

## Mechanistic models: SIR

### Compartments and notation

A **compartmental model** describes a population of interacting individuals under three assumptions:

- every individual is in exactly one state, its **compartment**, and people in the same compartment have the same modeled behavior;
- changes of state (i.e. of compartment) follow fixed transition rules;
- the population is **well mixed**: a susceptible person's chance of meeting an infectious person depends on the infectious count, rather than on a contact network. We assume the infection rate follows the **mass-action law**, proportional to the product of the susceptible and infectious counts.

SIR has three compartments, and at any instant:
- $S(t)$ — number of susceptible individuals.
- $I(t)$ — number of infected (and infectious) individuals.
- $R(t)$ — number of removed individuals (recovered with immunity, or dead).

We assume a closed population: $S + I + R = N$ (constant). New symbols:

- $\beta$ — **transmission rate**: the rate at which susceptible-infected contacts produce new infections. Units: $\text{(individuals)}^{-1} \cdot \text{time}^{-1}$.
- $\gamma$ — **removal rate**: the rate at which infectious individuals leave $I$. Units: $\text{time}^{-1}$. In the corresponding stochastic model, the mean infectious period is $1/\gamma$.
- $R_0 = \beta N / \gamma$ — **basic reproduction number**: average number of secondary infections from a single infected case in a fully-susceptible population. Dimensionless.

The compartment $R(t)$ and the reproduction number $R_0$ are different quantities despite their similar notation.

### From transition rules to the SIR equations

The rules are two reactions, $S \to I \to R$:

1. **Infection**, $S + I \to 2I$. It needs a susceptible and an infectious person to meet. By mass action the rate is proportional to the number of such pairs: $\beta S I$ new infections per unit time.
2. **Removal**, $I \to R$. It involves one person and no meeting, so the rate is $\gamma I$ removals per unit time.

Units check: $\beta S I$ is a number of individuals per unit time, and $S I$ is individuals squared, so $\beta$ has units of $\text{(individuals)}^{-1} \cdot \text{time}^{-1}$, as stated in the setup. $\gamma$ is a plain rate $\text{time}^{-1}$.

Assembling the rates, each reaction removes from one compartment what it adds to the next:
$$
\boxed{
\begin{aligned}
\dot S &= -\beta\, S\, I, \\
\dot I &= \beta\, S\, I - \gamma\, I, \\
\dot R &= \gamma\, I,
\end{aligned}
}
$$
with initial conditions $S(0)$, $I(0)$, $R(0)$. In the textbook problem these are given. In a real outbreak they are the hardest numbers to know: how many people are infected right now, and how many are already immune, is exactly what testing and serology try to measure.

When $S$ people remain susceptible, each infectious person causes infections at rate $\beta S$ and leaves the infectious compartment at rate $\gamma$. Removed people do not become susceptible again in this model.

Three immediate observations:

1. **Conservation:** $\dot S + \dot I + \dot R = 0$, so the population stays constant. That is the first thing to check in any implementation: if the total drifts, the transition terms do not balance.
2. **$R$ is passive:** the dynamics of $S$ and $I$ don't depend on $R$. So you can analyse the 2D $(S, I)$ system and read off $R$ from the conservation law.
3. **Growth condition:** $\dot I > 0$ when $I>0$ and $\beta S > \gamma$, or $S > \gamma/\beta = N/R_0$. The infectious count grows only while the susceptible count exceeds this threshold.

### Early growth near the disease-free state

At the start of an outbreak almost no one has been infected, so $I + R \ll N$. Write the susceptible count exactly as
$$
S = N - (I + R) = N\left(1 - \frac{I + R}{N}\right) \approx N
$$
and substitute it into the $I$ equation:
$$
\dot I = \beta N I - \gamma I.
$$
The factor $(I + R)/N$ is the fraction of the population infected so far. While that fraction is small, we can just drop it. What remains is linear in $I$:
$$
\dot I \approx (\beta N - \gamma)\, I,
\qquad\text{so}\qquad
I(t) \approx I_0\, e^{(\beta N - \gamma)t}.
$$
Here $r=\beta N-\gamma$ is the early growth rate. The approximation ceases to hold when enough people have been infected that $S$ is no longer close to $N$.

What a small outbreak does depends only on the sign of $r$:

- $\beta N > \gamma$: the infections grow exponentially.
- $\beta N < \gamma$: the infections die out exponentially.

The switch happens at $\beta N/\gamma=1$ and the approximation fails as the susceptible fraction falls.

### The basic reproduction number $R_0$

The early-growth analysis singled out the combination
$$
R_0 = \frac{\beta N}{\gamma}.
$$
In the fully susceptible limit, one infectious person causes new infections at rate $\beta N$ per unit time and stays infectious for a mean time $1/\gamma$. Their product, $\beta N/\gamma$, is the average number of secondary cases per primary case. The condition $R_0>1$ says that a small introduction initially grows in that limit when each case causes more than one new case on average.

- $R_0 < 1$: a small introduction declines exponentially.
- $R_0 > 1$: a small introduction initially grows exponentially.
- $R_0 = 1$: the **epidemic threshold**.

With fixed $\beta$ and $\gamma$, the **effective reproduction number** in this model is $R_t=\beta S(t)/\gamma=R_0S(t)/N$. Reducing contact or transmission lowers $\beta$, shortening the time a person can transmit increases the removal rate $\gamma$. Either change lowers transmission under the new conditions. Vaccination moves protected susceptible people into the removed compartment and lowers $S$. The simple formula for $R_t$ depends on **homogeneous mixing**, the assumption that contacts depend on compartment counts rather than on who meets whom, and on the assumption that removed people cannot be infected again.

> **Aside: estimating $R_0$.** One can fit a transmission model to case counts, but those counts depend on testing, reporting, and the delay between infection and observation. Estimating $R_0$ also needs, beyond the initial slope, an infectious-period assumption or information about the early level and reporting of infections. An estimate therefore belongs to a specified population, period, and model, rather than to a disease in every setting.

### Herd immunity threshold

How many people must be immune in advance for a small infection to decline? Infection grows while $R_t=R_0S/N>1$. Suppose $V_0$ people are fully protected by vaccination before an infection is introduced. With $I(0)\approx0$ and $R(0)=V_0$, the susceptible fraction is $S/N\approx1-v$, where $v=V_0/N$ is the proportion of vaccinated individuals in the population. Then $R_t\approx R_0(1-v)<1$ when
$$
v>1-1/R_0.
$$
This is the **herd immunity threshold** in homogeneous SIR: above it, a small introduction declines on average even though some people remain susceptible.

For the illustrative values $R_0=3$ and $R_0=15$, the thresholds are about 67% and 93%, respectively.

> **Aside: infections after the peak.** Crossing $S=N/R_0$ makes $I$ start to decline; it does not make $I$ vanish. People who are still infectious cause further cases. For $R_0=3$ with a nearly fully susceptible initial population, the threshold corresponds to 67% ever infected, while the final number of infected (calculation below) gives about 94%. The difference of 27 percentage points is called **overshoot**.

### Final-size relation

Divide the $S$ equation by the $R$ equation. Time drops out, and so does $I$:
$$
\frac{dS}{dR} = \frac{-\beta S I}{\gamma I} = -\frac{R_0}{N} S.
$$
This equation is linear in $S$ as a function of $R$. Integrating from the initial state gives $S(t)=S_0\exp[-(R_0/N)(R(t)-R(0))]$. Suppose $R(0)=0$ and $I_0>0$. At the end, $I(\infty)=0$ and $S(\infty)=N-R(\infty)$. If $I_0/N$ is negligible, then $S_0/N\approx1$, giving an approximate equation for the **final attack rate** $x=R(\infty)/N$:
$$
1 - x = e^{-R_0 x}.
$$
The approximate equation has a root at $x=0$. For $R_0>1$ it also has a positive root, which is the limit reached as a nonzero introduction becomes small relative to $N$. The positive-root estimates are $x\approx0.58$ for $R_0=1.5$, $0.797$ for $R_0=2$, and $0.94$ for $R_0=3$.

### Worked example 1: numerical integration

This cell integrates a population of one million with one initial infectious person and prints the peak and final removed fraction:

In [ ]:
import numpy as np
from scipy.integrate import odeint

def sir_rhs(y, t, beta, gamma):
    S, I, R = y
    dS = -beta * S * I
    dI =  beta * S * I - gamma * I
    dR =  gamma * I
    return [dS, dI, dR]

N = 1_000_000
I0 = 1
S0 = N - I0
R0_compartment = 0
beta, gamma = 0.3 / N, 0.1     # so R0 = beta * N / gamma = 3
t = np.linspace(0, 200, 1000)
sol = odeint(sir_rhs, [S0, I0, R0_compartment], t, args=(beta, gamma))
S, I, R = sol.T
print(f"Peak infections: {I.max():.0f} at day {t[I.argmax()]:.1f}")
print(f"Final attack rate: {R[-1]/N:.2%}")

The peak infectious count is about 300,000 near day 73, and the final removed fraction is about 94%. Early $I(t)$ follows an exponential approximation; after $S$ crosses $N/R_0$, $I$ declines. Check that $S+I+R=N$ at each output time: every infection and removal must transfer a person between compartments. This checks the implementation, while comparison with observed data would test whether its assumptions describe an outbreak. The full version in [`code/10_sir_epidemiology/sir_basic.py`](./code/10_sir_epidemiology/sir_basic.py) plots all three compartments and marks the peak threshold.

### Worked example 2: fitting SIR to daily case counts

Suppose we observe the number of *new* infections each day, rather than the cumulative total. For a one-day interval $[t,t+1]$, the SIR prediction is $S(t)-S(t+1)$. We generate independent Poisson counts around those daily predictions and fit $\beta N$ and $\gamma$ by maximum likelihood. The population size and initial infectious count are treated as known so that we can focus on the two rates.

In [ ]:
import numpy as np
from scipy.integrate import odeint
from scipy.optimize import minimize

N, I0 = 100_000, 5
days = np.arange(101)

def sir_rhs(y, t, beta, gamma):
    S, I, R = y
    return [-beta * S * I, beta * S * I - gamma * I, gamma * I]

def expected_cases(rates, time_grid):
    transmission, gamma = rates   # transmission = beta * N
    sol = odeint(sir_rhs, [N - I0, I0, 0], time_grid,
                 args=(transmission / N, gamma))
    return np.maximum(sol[:-1, 0] - sol[1:, 0], 1e-12)

rng = np.random.default_rng(0)
observed = rng.poisson(expected_cases([0.35, 0.15], days))

def fit(day_count):
    grid = days[:day_count + 1]
    counts = observed[:day_count]
    def nll(rates):
        predicted = expected_cases(rates, grid)
        return np.sum(predicted - counts * np.log(predicted))
    result = minimize(nll, [0.3, 0.1], method="L-BFGS-B",
                      bounds=[(1e-4, 2.0), (1e-4, 2.0)])
    assert result.success
    transmission, gamma = result.x
    return transmission, gamma, transmission / gamma

for label, rates in [("true", (0.35, 0.15, 0.35 / 0.15)),
                     ("first 25 days", fit(25)), ("all 100 days", fit(100))]:
    print(f"{label}: beta*N={rates[0]:.3f}, gamma={rates[1]:.3f}, R0={rates[2]:.3f}")

The first 25 days give $\widehat R_0=2.789$, while the full 100 days give $2.327$, close to the true value $2.333$. The short-window estimate is sensitive to noise and the assumption that $I_0$ is known. This likelihood uses daily counts because the Poisson draws in the example are independent *conditional on the deterministic curve*. Cumulative counts reuse earlier observations, so treating each cumulative total as an independent Poisson observation would count the same cases repeatedly. A real data analysis would also model reporting and observation delays.

The early exponential slope constrains $r=\beta N-\gamma$. The level of daily incidence supplies another constraint on $\beta N$ here because $I_0$ and reporting are assumed known. If either is unknown, that level can be rescaled, leaving many pairs $(\beta,\gamma)$ with the same slope. Later observations of the curve bending and declining help separate the rates. The fit on simulated data tests parameter recovery under our assumed observation model; it does not validate that observation model for real cases. The longer companion script is in [`code/10_sir_epidemiology/sir_fit.py`](./code/10_sir_epidemiology/sir_fit.py).

## Statistical models: autoregressive forecasts

### Predicting from recent observations

The SIR fit required a transmission model to account for observed counts. We can also start with an observed series, such as cumulative reported death counts $D(t)$, and extend its recent trend. Deaths have reporting delays and noise but can be less affected by case ascertainment than reported infections. The choice of series, its reporting process, and the forecast horizon still matter.

An **autoregressive** forecast updates the next value from earlier values of the same series. For equally spaced observations, a general rule has the form
$$
\widehat X(t+1)=f\big(X(t),X(t-1),\ldots\big).
$$
For a forecast beyond one step, predicted values replace future observations as inputs. The linear and exponential rules below fit their coefficients on a recent time window; each then updates its own predicted value through a lag-one recurrence.

**Linear Model.** Let $t=1$ denote the first day in a selected window. Fit a linear trend to its death counts, writing the fitted value as $\widehat D(t)=b_0+b_1(t-1)$. Advancing one day and subtracting gives
$$
\widehat D(t+1)-\widehat D(t)=b_1,
\qquad\text{or}\qquad
\boxed{\widehat D(t+1)=\widehat D(t)+b_1.}
$$
Thus the forecast adds the same **absolute increment** each day, starting from $\widehat D(1)=b_0$. This is a lag-one autoregressive update of predicted values with coefficient one and intercept $b_1$.

**Exponential Model.** We can also fit a line to the logarithms instead: $\log\widehat D(t)=e_0+e_1(t-1)$, or $\widehat D(t)=\exp[e_0+e_1(t-1)]$. The daily difference on the log scale is $e_1$, so
$$
\log\widehat D(t+1)-\log\widehat D(t)=e_1,
\qquad\text{or}\qquad
\boxed{\widehat D(t+1)=e^{e_1}\widehat D(t).}
$$
This forecast has a constant **multiplicative factor** $e^{e_1}$ per day, starting from $\widehat D(1)=e^{e_0}$. The coefficients $b_i$ and $e_i$ describe fitted trends and they are different from the transmission rate $\beta$.

The two rules can be refitted as new observations arrive, then iterated from the end of the fitting window for short forecasts. They contain no susceptible depletion, so neither can anticipate a change in epidemic phase from its fixed update rule. Linear and exponential components were used in county-level COVID-19 death forecasting by Altieri et al. (Further reading). In the next example we apply the same mathematics to cumulative infections $C(t)$.

### Worked example 3: extrapolating short-window trends

How well does each regression predict ten days beyond a ten-day window? We use the cumulative count $C=I+R$ generated by SIR so we know the future values against which to check the forecasts. The cell fits a line to $C$ and another to $\log C$, once during early growth and once near the peak:

In [ ]:
import numpy as np
from scipy.integrate import odeint

def sir_rhs(y, t, beta, gamma):
    S, I, R = y
    return [-beta * S * I, beta * S * I - gamma * I, gamma * I]

N, I0 = 1_000_000, 1
beta, gamma = 0.3 / N, 0.1
days = np.arange(0, 121)
sol = odeint(sir_rhs, [N - I0, I0, 0], days, args=(beta, gamma))
C = sol[:, 1] + sol[:, 2]

for start in [20, 65]:
    window = np.arange(start, start + 10)
    b1, b0 = np.polyfit(days[window], C[window], 1)
    e1, e0 = np.polyfit(days[window], np.log(C[window]), 1)
    t10 = days[start + 19]
    print(f"day {t10}: true {C[t10]:.0f}, linear {b0 + b1 * t10:.0f}, "
          f"exponential {np.exp(e0 + e1 * t10):.0f}")

In the early window (days 20–29), the fitted exponential daily multiplier is $e^{e_1}=1.222$, close to $e^r=e^{0.2}$. At day 39 the true cumulative count is 3,644; the exponential forecast is 3,675 and the linear forecast is 879. For the window on days 65–74, the day-84 count is 860,655, while the linear and exponential forecasts are 1,081,996 and 1,490,685. Both now overshoot because their fixed update rules do not anticipate the bending of the cumulative curve. The exponential model works well for the early ten-day forecast and poorly for the later one. These are checks against a simulated SIR trajectory, not validation against observed epidemic data. [`code/10_sir_epidemiology/autoregressive.py`](./code/10_sir_epidemiology/autoregressive.py) prints the intervening forecasts.

## Intuition

> **If $R_0>1$, why can infections be declining today?** $R_0$ describes transmission in a fully susceptible population under specified conditions. At time $t$, SIR uses $R_t=R_0S(t)/N$. Prior infections or vaccination can reduce $S$ until $R_t<1$, even though $R_0$ remains above 1. The sign of $\dot I$ depends on the *current* susceptible count, not on $R_0$ alone.

> **Is the herd immunity threshold the fraction eventually infected?** No. The threshold $1-1/R_0$ is the immune fraction at which a *small new introduction* stops growing in homogeneous SIR. During an outbreak, crossing $S=N/R_0$ only marks the peak of $I(t)$: people already infectious continue to cause new cases. With $R_0=3$, the peak occurs when roughly 67% have been infected, but the final-size equation gives roughly 94%. The difference is the overshoot described above.

> **Can the early growth rate determine $R_0$?** Near the disease-free state, $I(t)\approx I_0e^{rt}$ with $r=\beta N-\gamma$. Many pairs of $\beta$ and $\gamma$ give the same $r$ but different $R_0=\beta N/\gamma$. In Worked example 2, the *level* of daily incidence adds information because $I_0$ and reporting are assumed known. If either is uncertain, the same early slope supports more than one reproduction-number estimate.

> **How do the two statistical updates differ?** The linear rule adds the fitted increment $b_1$ each day; the exponential rule multiplies by $e^{e_1}$ each day. Both update a forecast of the observed series, and neither coefficient is the transmission rate $\beta$. In the early window of Worked example 3, the exponential multiplier agrees with the SIR growth approximation. Later, susceptible depletion changes the trend. At day 84 both forecasts exceed the population of one million, which a cumulative infection count cannot do.

## Connections

- **Builds on:** ODE integration for the SIR trajectory and maximum likelihood from Chapter 2 for fitting its rates to daily counts.
- **Related ideas:** dimensional analysis (Appendix C) explains why $R_0$ governs SIR after population and time are rescaled. Statistical forecasting uses fitted relationships between observations, while SIR uses transition rules. Models with exposed compartments or different contact patterns can extend those rules when homogeneous SIR misses important behavior.

## Exercises

1. **Conceptual.** A friend says, "If $R_0=3$, every infectious person infects exactly three others throughout the outbreak." Identify three problems with that statement.

2. **Derivation.** Suppose a fraction $v$ of an otherwise susceptible population is fully immune before one infectious person enters. Derive the condition on $v$ for the infectious count to decline initially. Evaluate it for $R_0=3$. Explain why that percentage need not equal the final attack rate of an unmitigated outbreak.

3. **Computational.** For $R_0\in\{1.2,1.5,2.0,3.0,5.0,10.0\}$, find the *positive* root of $1-x=e^{-R_0x}$ and plot the final attack rate $x$ versus $R_0$. Why is the zero root of the approximate equation not the relevant one for a small but nonzero introduction?

4. **Computational.** Integrate the SIR equations for $R_0\in\{1.5,2.5,4.0\}$ with $N=10^6$, $I_0=1$, and $\gamma=0.1$ per day. For each value, report the peak infectious fraction and day, and check whether $S/N\approx1/R_0$ at the peak. Choose a time interval long enough to include all three peaks.

5. **Computational.** Generate independent Poisson daily case counts around a simulated SIR curve with known $\beta$ and $\gamma$. Fit the rates from an early window and from the full curve. Compare both estimates of $R_0$ with its true value. Explain why the early incidence *level* provides information beyond its growth slope in this example, and name an assumption that makes that extra information available.

6. **Computational.** A fitted linear forecast starts at $\widehat D(1)=20$ and adds 4 per day; a fitted exponential forecast starts at $\widehat D(1)=20$ and has daily multiplier 1.2. Compute both forecasts at day 4. Then use Worked example 3 to identify which rule is closer to the simulated count at day 39 and which is closer at day 84. Why does the answer change?

7. **Modeling judgment.** A health authority reports that its estimated reproduction number under current conditions fell from 3.0 to 1.2 after a mask policy and says the region has "almost achieved herd immunity." What distinction between $R_0$ and $R_t$ would you ask them to clarify? Name two checks before attributing the change to masks and one other explanation for falling reported cases.

:::{admonition} Solutions
:class: dropdown

**1.** $R_0$ is an average, so individuals can cause different numbers of infections. It refers to the fully susceptible limit; later in SIR the current value is $R_t=R_0S/N$. It also depends on the contact conditions used to define $\beta$, which need not stay fixed. None of these facts supports "exactly three others throughout."

**2.** Initially $S/N\approx1-v$, so $R_t\approx R_0(1-v)$. The infectious count declines when $R_t<1$, giving $v>1-1/R_0$. For $R_0=3$ this means *more than* two thirds of the population must be immune; at exactly two thirds the initial growth rate is zero in the approximation. In an unmitigated epidemic with $R_0=3$, infections continue after the peak and the final attack rate is about 94%, rather than 67%. The threshold concerns initial growth after immunity is present; the final size concerns the entire epidemic.

**3.** See [`code/10_sir_epidemiology/final_size.py`](./code/10_sir_epidemiology/final_size.py). The positive roots are approximately $0.314$, $0.583$, $0.797$, $0.940$, $0.993$, and $0.99995$ for the six listed $R_0$ values. The zero root appears when $I_0/N$ is set to zero in the approximate final-size equation. For a finite $I_0>0$, the exact equation is $1-x=(S_0/N)e^{-R_0x}$ with $S_0<N$, so $x=0$ is not a solution. The positive root approaches the final attack rate as the initial infected fraction becomes small.

**4.** See [`code/10_sir_epidemiology/sir_basic.py`](./code/10_sir_epidemiology/sir_basic.py). For $R_0=1.5$, $2.5$, and $4.0$, the peak fractions are about $0.063$, $0.233$, and $0.403$, at days $253$, $95$, and $50$, respectively. At each peak, $\dot I=0$ while $I>0$, so the equations require $S/N=1/R_0$. A run that ends before day 253 would mistake its last output point for the first peak.

**5.** See [`code/10_sir_epidemiology/sir_fit.py`](./code/10_sir_epidemiology/sir_fit.py). In the seeded example, true $R_0=2.333$, the first-25-day estimate is $2.789$, and the full-curve estimate is $2.327$. The early slope constrains $\beta N-\gamma$; the early count level also depends on $\beta N I_0$. Because $I_0$ and reporting are fixed in this example, the level adds information about $\beta N$. Unknown initial prevalence or reporting would weaken that inference. An early fit need not recover $R_0$ accurately merely because it finds the growth slope.

**6.** Three updates take day 1 to day 4. The linear forecast is $20+3(4)=32$ and the exponential forecast is $20(1.2)^3=34.56$. At day 39, the exponential forecast is 3,675 against a simulated count of 3,644, much closer than the linear forecast of 879. At day 84, the linear forecast is 1,081,996 against 860,655, closer than the exponential forecast of 1,490,685, although both overshoot. Both forecasts also exceed the population of one million. The early cumulative curve is nearly exponential; later its growth slows as $S$ falls. Neither fixed update rule predicts that change or enforces the population limit. [`code/10_sir_epidemiology/autoregressive.py`](./code/10_sir_epidemiology/autoregressive.py) prints both forecast windows.

**7.** $R_0$ refers to the fully susceptible limit under specified transmission conditions, while $R_t$ includes the current susceptible fraction. A current estimate of 1.2 does not establish that immunity caused the decline, and $R_t=1.2$ still allows $I$ to grow in SIR. Check whether testing or reporting changed, and whether the timing of the estimate is consistent with the mask policy after reporting delays. Prior infections could also have reduced $S$ and therefore $R_t$ without changing $\beta$. Case counts alone cannot separate these explanations.

:::

## Further reading

- Kermack and McKendrick (1927), *Proc. Roy. Soc. A* 115(772):700–721 — the original derivation of a compartmental epidemic model. [doi:10.1098/rspa.1927.0118](https://doi.org/10.1098/rspa.1927.0118).
- Anderson and May, *Infectious Diseases of Humans* (Oxford, 1991) — a detailed treatment of reproduction numbers, thresholds, and population structure. ISBN 9780198545996 · [HOLLIS](https://hollis.harvard.edu/discovery/search?query=any,contains,Infectious%20Diseases%20of%20Humans%20Anderson%20May&tab=LibraryCatalog&search_scope=MyInstitution&vid=01HVD_INST:HVD2&offset=0).
- Keeling and Rohani, *Modeling Infectious Diseases in Humans and Animals* (Princeton, 2008) — computational extensions of SIR, including more detailed contact patterns. ISBN 9780691116174 · [HOLLIS](https://hollis.harvard.edu/discovery/search?query=any,contains,Modeling%20Infectious%20Diseases%20in%20Humans%20and%20Animals%20Keeling%20Rohani&tab=LibraryCatalog&search_scope=MyInstitution&vid=01HVD_INST:HVD2&offset=0) · [companion code](https://warwick.ac.uk/fac/cross_fac/zeeman_institute/staffv2/keeling/mid/).
- Barrat, Barthélemy, and Vespignani, *Dynamical Processes on Complex Networks* (Cambridge, 2008) — Chapter 9 studies epidemic spreading when contacts follow a network rather than SIR's well-mixed assumption. ISBN 9780521879507 · [publisher](https://www.cambridge.org/core/books/dynamical-processes-on-complex-networks/D0173F07E0F05CEE9CF7A6BDAF48E9FC) · [HOLLIS](https://hollis.harvard.edu/discovery/search?query=any,contains,Dynamical%20Processes%20on%20Complex%20Networks%20Barrat&tab=LibraryCatalog&search_scope=MyInstitution&vid=01HVD_INST:HVD2&offset=0).
- Altieri et al., "Curating a COVID-19 data repository and forecasting county-level death counts in the United States," *Harvard Data Science Review* (2021) — an example of short-window linear and exponential forecasts combined with other predictions. [doi:10.1162/99608f92.1d4e0dae](https://doi.org/10.1162/99608f92.1d4e0dae) · [arXiv:2005.07882](https://arxiv.org/abs/2005.07882).